# Home Credit nested training
Run the benchmark first, then complete the suite. All outputs are saved to Drive. No GPU is required. Enter your repository URL and data folder below.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPOSITORY.git"
DATA_DIR = "/content/drive/MyDrive/COMP90051/pretraining"
RESULTS_DIR = "/content/drive/MyDrive/COMP90051/training_results"


In [ ]:
import subprocess
from pathlib import Path
import shutil
repo = Path("/content/comp90051_project")
if not repo.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
subprocess.run(["python", "-m", "pip", "install", "-r", str(repo / "training/requirements.txt")], check=True)
local_data = Path("/content/homecredit_inputs")
local_data.mkdir(exist_ok=True)
for name in ["features_compact.csv", "labels.csv", "experiments_compact.json", "nested_folds.csv"]:
    shutil.copy2(Path(DATA_DIR) / name, local_data / name)


## Benchmark: one outer fold
This performs model tuning and outer evaluation for fold 0. Use the time and memory to decide the final workload.

In [ ]:
base = ["python", "-m", "training.run", "--data-dir", str(local_data),
        "--output-dir", RESULTS_DIR, "--threads", "2"]
subprocess.run(base + ["--experiments", "E9_dynamics24", "--models", "lr", "cart", "lightgbm",
                       "--outer-folds", "0"], cwd=repo, check=True)


## Full run
Execute this cell only when ready. Completed fits are skipped. You can reduce the experiment list or run one model at a time. A runtime interruption does not erase completed Drive checkpoints.

In [ ]:
subprocess.run(base + ["--experiments", "E0", "E1", "E4", "E7_24m", "E8_multiscale24", "E9_dynamics24",
                       "--models", "lr", "cart", "lightgbm"], cwd=repo, check=True)


In [ ]:
import pandas as pd
summary = pd.read_csv(Path(RESULTS_DIR) / "summary.csv")
display(summary[summary.metric.isin(["roc_auc", "average_precision", "log_loss"])])
